In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 98.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import json
import re

## ПРОВЕРИМ ТОЧНОСТЬ НА ТЕСТОВОМ ДАТАСЕТЕ БЕЗ ДООБУЧЕНИЯ

In [3]:
data_2000 = pd.read_csv("df_2000_all_new.csv")
data_2000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1488,0,• Объем торгов металлами в феврале составил 61...
1802,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
1803,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1489,1,"""​​Что означает 100 рублей за доллар для префо..."
1804,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...


In [4]:
## ТУТ ТОЖЕ КОСЯКА НЕ БЫЛО)
data_2000 = data_2000[data_2000['doc_id'] != 'up']
data_2000['doc_id'] = data_2000['doc_id'].astype('int')

test = data_2000[(data_2000['doc_id'] > 1234)]

/tmp/ipykernel_1244/733867683.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_2000['doc_id'] = data_2000['doc_id'].astype('int')


In [5]:
dataset = list(test['original_text'])

## ТЕСТ МОДЕЛИ БЕЗ ОБУЧЕНИЯ

In [6]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16, # или torch.float16
    device_map="auto"           # автоматически распределит веса на GPU
)

responces = []
with torch.no_grad():
    for i, text in enumerate(dataset, 1):
        print(f'Обработка документа {i}...')


        prompt = ( "Твоя задача — анализировать новости "
        "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
        "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
        "Ответь строго в формате JSON без вступлений и пояснений: "
        "{"
        "\"процентная ставка\": \"тип изменения\","
        "\"ВВП\": \"тип изменения\","
        "\"инфляция\": \"тип изменения\","
        "\"безработица\": \"тип изменения\","
        "\"капитал\": \"тип изменения\","
        "\"инвестиции\": \"тип изменения\","
        "\"производство\": \"тип изменения\","
        "\"потребление\": \"тип изменения\","
        "\"численность рабочей силы\": \"тип изменения\","
        "\"сбережения\": \"тип изменения\","
        "\"заработные платы\": \"тип изменения\","
        "\"доходы населения\": \"тип изменения\","
        "\"валютный курс\": \"тип изменения\","
        "\"импорт\": \"тип изменения\","
        "\"экспорт\": \"тип изменения\","
        "\"государственные расходы\": \"тип изменения\","
        "\"государственный долг\": \"тип изменения\","
        "\"дефицит бюджета\": \"тип изменения\""
        "} "
        f"Текст новости: {text}"
    )
        messages = [
            {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        responces.append(response)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Обработка документа 1...
Обработка документа 2...
Обработка документа 3...
Обработка документа 4...
Обработка документа 5...
Обработка документа 6...
Обработка документа 7...
Обработка документа 8...
Обработка документа 9...
Обработка документа 10...
Обработка документа 11...
Обработка документа 12...
Обработка документа 13...
Обработка документа 14...
Обработка документа 15...
Обработка документа 16...
Обработка документа 17...
Обработка документа 18...
Обработка документа 19...
Обработка документа 20...
Обработка документа 21...
Обработка документа 22...
Обработка документа 23...
Обработка документа 24...
Обработка документа 25...
Обработка документа 26...
Обработка документа 27...
Обработка документа 28...
Обработка документа 29...
Обработка документа 30...
Обработка документа 31...
Обработка документа 32...
Обработка документа 33...
Обработка документа 34...
Обработка документа 35...
Обработка документа 36...
Обработка документа 37...
Обработка документа 38...
Обработка документа 3

In [7]:
## СОХРАРНИМ РЕЗУЛЬТАТЫ НЕ ОБУЧЕННОГО КВЕНА
df_qwen_untr = [{'responce': responces[i], 'doc_id': test.iloc[i, -3],
      'chunk_id': test.iloc[i, -2], 'original_text': test.iloc[i, -1]} for i in range(len(responces))]
df_qwen_untr = pd.DataFrame(df_qwen_untr)
df_qwen_untr

,responce,doc_id,chunk_id,original_text
0,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1236,0,"В СМИ звучали предположения, что они имели отн..."
1,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1238,0,"""​​Продажа Яндекса: чего ждать держателям акци..."
2,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1239,0,"""Топ-30 российских компаний по капитализации. ..."
3,"{""процентная ставка"": ""up"",""ВВП"": ""not stated""...",1239,1,"""​​Рынки обеспокоены растущими ценами на нефть..."
4,"```json\n{""процентная ставка"": ""not stated"",""В...",1239,2,• Рынки Азиатско-Тихоокеанского региона демонс...
...,...,...,...,...
263,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1488,0,• Объем торгов металлами в феврале составил 61...
264,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
265,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1489,1,"""​​Что означает 100 рублей за доллар для префо..."
266,"{""процентная ставка"": ""not stated"",""ВВП"": ""not...",1491,0,В начале октября валютный курс тестирует сопро...


In [8]:
df_qwen_untr.to_csv('df_qwen_untr_test.csv', index=False)

In [12]:
df_to_check = test
# ------------------------------------------

expected_keys = [
    "процентная ставка", "ВВП", "инфляция", "безработица", "капитал",
    "инвестиции", "производство", "потребление", "численность рабочей силы",
    "сбережения", "заработные платы", "доходы населения", "валютный курс",
    "импорт", "экспорт", "государственные расходы", "государственный долг",
    "дефицит бюджета"
]

total_accuracy = 0
valid_json_count = 0
up_down_accuracy = 0
k = len(responces) # Берем по количеству реально полученных ответов

not_up_down = 0

print("--- Начинаю робастный расчет метрик ---")

for i, pred_raw in enumerate(responces):
    try:
        clean_pred = pred_raw.strip()
        json_match = re.search(r'\{.*\}', clean_pred, re.DOTALL)

        if not json_match:
            print(f"Документ {i+1}: JSON не найден в тексте")
            continue

        pred_json = json.loads(json_match.group())

        correct_in_doc = 0
        correct_in_doc_up_down = 0
        cur_up_down_ans = 0
        for entity in expected_keys:
            # ответ модели
            model_val = str(pred_json.get(entity, "missing")).lower().strip()


            true_val = str(df_to_check.iloc[i][entity]).lower().strip()

            if model_val == true_val:
                correct_in_doc += 1

            ## отдельно считаем метрики для not_stated(ЕСТЬ ДИЗБАЛАНС КЛАССОВ)
            if true_val != 'not stated':
                cur_up_down_ans += 1
                if model_val == true_val:
                    correct_in_doc_up_down += 1
        ## ПРОВЕРЯЕМ ЕСТЬ ЛИ ХОТЯ БЫ ОДИН UP/DOWN В ОТВЕТАХ
        if cur_up_down_ans != 0:
            up_down_accuracy += correct_in_doc_up_down / cur_up_down_ans
        else:
            ## СЧИТАЕМ ЧИСЛО ПРАВИЛЬНЫХ ОТВЕТОВ ГДЕ ВСЕХ СУЩНОСТЕЙ НЕТ
            not_up_down += 1

        total_accuracy += (correct_in_doc / len(expected_keys))

        valid_json_count += 1

    except json.JSONDecodeError:
        print(f"Документ {i+1}: Ошибка декодирования JSON")
    except KeyError as e:
        print(f"Документ {i+1}: В датафрейме нет колонки {e}")
    except Exception as e:
        print(f"Документ {i+1}: Непредвиденная ошибка {type(e).__name__}: {e}")

final_accuracy = total_accuracy / k if k > 0 else 0
accuracy_up_down = up_down_accuracy / (k - not_up_down)

print("-" * 30)
print(f"ИТОГОВАЯ ACCURACY: {final_accuracy:.4f}")
print(f"ИТОГОВАЯ ACCURACY НА ПРИМЕРАХ UP/DOWN: {accuracy_up_down:.4f}")


print(f"Успешно обработано: {valid_json_count} из {k}")


--- Начинаю робастный расчет метрик ---
------------------------------
ИТОГОВАЯ ACCURACY: 0.9177
ИТОГОВАЯ ACCURACY НА ПРИМЕРАХ UP/DOWN: 0.5185
Успешно обработано: 268 из 268


## КАЧЕСТВО УЖЕ НЕПЛОХОЕ 92% на ВСЕХ ПРИМЕРАХ И 52% НА ПРИМЕРАХ С ИЗМЕНЕНИЯМИ, НО МЫ ХОТИМ ЛУЧШЕ

---



